# 4. Vanna Training

This notebook trains Vanna with the schema and sample queries for the EMR database.

In [1]:
import os
import sys

# Ultimate foolproof path injection to find 'src' regardless of current working directory
cwd = os.path.abspath(os.getcwd())
project_root = None

# 1. Walk up from current working directory
temp = cwd
for _ in range(4):
    if os.path.exists(os.path.join(temp, 'src', 'config.py')):
        project_root = temp
        break
    parent = os.path.abspath(os.path.join(temp, '..'))
    if parent == temp:
        break
    temp = parent

# 2. Fallback to searching sys.path
if not project_root:
    for path in list(sys.path):
        if not path:
            continue
        candidate = os.path.abspath(path)
        for folder in [candidate, os.path.abspath(os.path.join(candidate, '..')), os.path.abspath(os.path.join(candidate, '..', '..'))]:
            if os.path.exists(os.path.join(folder, 'src', 'config.py')):
                project_root = folder
                break
        if project_root:
            break

if project_root:
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
    path_prefix = os.path.relpath(project_root, cwd)
    print(f"Project root found: {project_root}")
    print(f"Current working directory: {cwd}")
    print(f"Relative path prefix: {path_prefix}")
else:
    print("Warning: Could not automatically detect project root. Using default path prefix '..'")
    path_prefix = ".."

from src.services.providers import get_vanna

Project root found: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator
Current working directory: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator\notebook
Relative path prefix: ..


d:\PEMROGRAMAN\LLM-DESKTOP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("1. Initializing Vanna...")
# Ensure PostgreSQL is running and notebook 3 has been executed
vn = get_vanna()

1. Initializing Vanna...


In [3]:
print("2. Training Vanna with DDL...")
ddl = """
CREATE TABLE emr_records (
    status VARCHAR,
    emr_name VARCHAR,
    serial_number VARCHAR,
    machine_product VARCHAR,
    machine_model VARCHAR,
    account_account_name VARCHAR,
    sub_call_type VARCHAR,
    pmact_type VARCHAR,
    subjects TEXT,
    symptom TEXT,
    caused_of_problem TEXT,
    action_how_was_problem_corrected TEXT,
    branch_site VARCHAR,
    part_suply VARCHAR,
    main_cause_part_no VARCHAR,
    part_description VARCHAR,
    part_description_1 VARCHAR,
    techcare_component VARCHAR,
    techcare_component_1 VARCHAR,
    techcare_sub_component VARCHAR,
    techcare_sub_component_1 VARCHAR,
    created_date TIMESTAMP,
    emr_last_closed_date TIMESTAMP,
    model_family VARCHAR,
    graph_community_summary TEXT
);
"""
vn.train(ddl=ddl)

2. Training Vanna with DDL...
Adding ddl: 
CREATE TABLE emr_records (
    status VARCHAR,
    emr_name VARCHAR,
    serial_number VARCHAR,
    machine_product VARCHAR,
    machine_model VARCHAR,
    account_account_name VARCHAR,
    sub_call_type VARCHAR,
    pmact_type VARCHAR,
    subjects TEXT,
    symptom TEXT,
    caused_of_problem TEXT,
    action_how_was_problem_corrected TEXT,
    branch_site VARCHAR,
    part_suply VARCHAR,
    main_cause_part_no VARCHAR,
    part_description VARCHAR,
    part_description_1 VARCHAR,
    techcare_component VARCHAR,
    techcare_component_1 VARCHAR,
    techcare_sub_component VARCHAR,
    techcare_sub_component_1 VARCHAR,
    created_date TIMESTAMP,
    emr_last_closed_date TIMESTAMP,
    model_family VARCHAR,
    graph_community_summary TEXT
);



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4945.12it/s]


'730578ce-3c10-52eb-97d6-2ce978292680-ddl'

In [ ]:
print("3. Training Vanna with Documentation...")
doc = """
The 'emr_records' table contains Equipment Maintenance Records for heavy machinery.

CRITICAL COLUMN USAGE RULES:
- 'machine_model' is the SPECIFIC equipment model with suffix (e.g., 'HD785-7', 'PC200-10M0', 'D155A-6'). 
  ALWAYS use this column when the user mentions a model name with a dash and number suffix like HD785-7 or PC200-8.
- 'model_family' is the BROAD category WITHOUT suffix (e.g., 'HD785', 'PC200', 'D155A'). 
  Only use this column when the user asks about a general family like "semua model PC200" without specifying the exact variant.
- 'branch_site' is the location where the equipment is operated.
- 'graph_community_summary' contains the executive summary from the GraphRAG pipeline. 
  When filtering for records that HAVE a GraphRAG summary, use: graph_community_summary IS NOT NULL.
  When searching for specific topics in the summary, use: graph_community_summary ILIKE '%keyword%'.
- 'techcare_component' is the main component or subsystem where the failure occurred (e.g., 'FINAL DRIVE', 'SWING MOTOR').\n  Use this when the user asks about a specific component or wants to know which components fail most often.\n- 'techcare_sub_component' is the specific sub-part of the component (e.g., 'Floating Seal', 'Injector').\n- Use 'created_date' when filtering by month or year.
"""
vn.train(documentation=doc)


3. Training Vanna with Documentation...
Adding documentation....


'8e92e3a1-847b-5b45-afc7-f602e4d0f41e-doc'

In [ ]:
print("4. Training Vanna with Sample Q&A...")

print("4. Training Vanna with Sample Q&A...")

qa_pairs = [
    ("Berapa total fault per model?", "SELECT machine_model, COUNT(*) as total FROM emr_records GROUP BY machine_model ORDER BY total DESC;"),
    ("Model apa yang paling sering rusak?", "SELECT machine_model, COUNT(*) as total FROM emr_records GROUP BY machine_model ORDER BY total DESC LIMIT 1;"),
    ("Tampilkan 5 site dengan masalah terbanyak", "SELECT branch_site, COUNT(*) as total FROM emr_records GROUP BY branch_site ORDER BY total DESC LIMIT 5;"),
    ("Tampilkan tren kerusakan per bulan", "SELECT DATE_TRUNC('month', created_date) as month, COUNT(*) as total FROM emr_records GROUP BY month ORDER BY month;"),
    ("Tampilkan tren masalah pada PC200 per bulan", "SELECT DATE_TRUNC('month', created_date) as month, COUNT(*) as total FROM emr_records WHERE model_family = 'PC200' GROUP BY month ORDER BY month;"),
    ("berikan perbandingan antara unit yang sering bermasalah dengan yang paling tidak sering bermasalah", "(SELECT machine_model, COUNT(*) as total_faults, 'Paling Sering' as status FROM emr_records GROUP BY machine_model ORDER BY total_faults DESC LIMIT 5) UNION ALL (SELECT machine_model, COUNT(*) as total_faults, 'Paling Jarang' as status FROM emr_records GROUP BY machine_model ORDER BY total_faults ASC LIMIT 5) ORDER BY status DESC, total_faults DESC;"),
    ("perbandingan unit paling sering rusak dengan unit yang paling jarang rusak", "(SELECT machine_model, COUNT(*) as total_faults, 'Paling Sering' as status FROM emr_records GROUP BY machine_model ORDER BY total_faults DESC LIMIT 5) UNION ALL (SELECT machine_model, COUNT(*) as total_faults, 'Paling Jarang' as status FROM emr_records GROUP BY machine_model ORDER BY total_faults ASC LIMIT 5) ORDER BY status DESC, total_faults DESC;"),
    ("Apa tema kerusakan utama (GraphRAG summary) untuk masalah pada alat PC200?", "SELECT DISTINCT graph_community_summary FROM emr_records WHERE model_family = 'PC200' AND graph_community_summary IS NOT NULL;"),
    ("Hitung berapa banyak kasus kerusakan yang berkaitan dengan kluster Sistem Pendingin (Cooling System) menurut GraphRAG", "SELECT COUNT(*) FROM emr_records WHERE graph_community_summary ILIKE '%Cooling System%' OR graph_community_summary ILIKE '%Sistem Pendingin%';"),
    ("Berapa banyak kasus kerusakan pada model HD785-7 yang memiliki ringkasan GraphRAG (dimana kolom graph_community_summary tidak kosong)?", "SELECT COUNT(*) FROM emr_records WHERE machine_model = 'HD785-7' AND graph_community_summary IS NOT NULL;")
]

for q, sql in qa_pairs:
    vn.train(question=q, sql=sql)

print("Training complete.")


for q, sql in qa_pairs:
    vn.train(question=q, sql=sql)

print("Training complete.")

4. Training Vanna with Sample Q&A...
4. Training Vanna with Sample Q&A...
Training complete.
Training complete.


In [7]:
print("5. Testing SQL Generation...")
# This will use the local Ollama LLM defined in our custom Vanna implementation
test_query = "Hitung berapa banyak kasus kerusakan yang berkaitan dengan kluster Sistem Pendingin (Cooling System) menurut GraphRAG"
sql = vn.generate_sql(test_query)
print(f"Question: {test_query}\nGenerated SQL: {sql}")

if sql:
    df = vn.run_sql(sql)
    print(df)

5. Testing SQL Generation...
Question: Hitung berapa banyak kasus kerusakan yang berkaitan dengan kluster Sistem Pendingin (Cooling System) menurut GraphRAG
Generated SQL: SELECT
Empty DataFrame
Columns: []
Index: [0]
